In [ ]:
import numpy as np

class AntColonyOptimizationTSP:
    def __init__(self, distances, n_ants, n_best, n_iterations, decay, alpha=1.0, beta=2.0, Q=100.0):
        """
        Args:
            distances (2D array): Square matrix of pairwise distances between cities.
            n_ants (int): Number of ants running per iteration.
            n_best (int): Number of top ants that deposit pheromones (Rank-based/Elitist flavor).
            n_iterations (int): Maximum number of iterations.
            decay (float): Evaporation rate (rho), between 0 and 1.
            alpha (float): Relative importance of pheromone (history).
            beta (float): Relative importance of heuristic (visibility/distance).
            Q (float): Total pheromone deposition constant.
        """
        self.distances  = distances
        self.pheromone  = np.ones(self.distances.shape) / len(distances)
        self.all_inds   = range(len(distances))
        self.n_ants     = n_ants
        self.n_best     = n_best
        self.n_iterations = n_iterations
        self.decay      = decay
        self.alpha      = alpha
        self.beta       = beta
        self.Q          = Q

    def run(self):
        shortest_path = None
        all_time_shortest_path = ("placeholder", np.inf)

        for i in range(self.n_iterations):
            all_paths = self.gen_all_paths()
            self.spread_pheromone(all_paths, self.n_best, shortest_path=shortest_path)
            shortest_path = min(all_paths, key=lambda x: x[1])

            if shortest_path[1] < all_time_shortest_path[1]:
                all_time_shortest_path = shortest_path

            self.pheromone *= (1 - self.decay)
            print(f"Iteration {i+1}/{self.n_iterations}: Best Distance = {shortest_path[1]:.2f}")

        return all_time_shortest_path

    def spread_pheromone(self, all_paths, n_best, shortest_path):
        sorted_paths = sorted(all_paths, key=lambda x: x[1])
        for path, dist in sorted_paths[:n_best]:
            for move in path:
                self.pheromone[move] += self.Q / dist

    def gen_path_dist(self, path):
        total_dist = 0
        for ele in path:
            total_dist += self.distances[ele]
        return total_dist

    def gen_all_paths(self):
        all_paths = []
        for i in range(self.n_ants):
            path = self.gen_path(0)
            all_paths.append((path, self.gen_path_dist(path)))
        return all_paths

    def gen_path(self, start):
        path = []
        visited = set()
        visited.add(start)
        prev = start

        for i in range(len(self.distances) - 1):
            move = self.pick_move(self.pheromone[prev], self.distances[prev], visited)
            path.append((prev, move))
            int_move = int(move)
            visited.add(int_move)
            prev = move

        path.append((prev, start))
        return path

    def pick_move(self, pheromone, dist, visited):
        pheromone = np.copy(pheromone)
        pheromone[list(visited)] = 0

        with np.errstate(divide='ignore'):
            eta = 1.0 / dist
        eta[list(visited)] = 0

        row = (pheromone ** self.alpha) * (eta ** self.beta)

        norm_row = row / row.sum()
        move = np.random.choice(self.all_inds, p=norm_row)
        return move


if __name__ == "__main__":


    city_distances = np.array([
        [10, 21, 25, 14, 0]
    ])

    city_names = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

    aco = AntColonyOptimizationTSP(
        distances=city_distances,
        n_ants=10,
        n_best=3,
        n_iterations=20,
        decay=0.2,
        alpha=1.0,
        beta=2.0
    )

    best_route, min_distance = aco.run()

    readable_route = [city_names[step[0]] for step in best_route] + [city_names[best_route[-1][1]]]

    print("\n" + "="*40)
    print("OPTIMIZED OUTPUT")
    print("="*40)
    print(f"Shortest Route Found: {' -> '.join(readable_route)}")
    print(f"Minimum Total Travel Distance: {min_distance} units")


Iteration 1/20: Best Distance = 10.00
Iteration 2/20: Best Distance = 10.00
Iteration 3/20: Best Distance = 10.00
Iteration 4/20: Best Distance = 10.00
Iteration 5/20: Best Distance = 10.00
Iteration 6/20: Best Distance = 10.00
Iteration 7/20: Best Distance = 10.00
Iteration 8/20: Best Distance = 10.00
Iteration 9/20: Best Distance = 10.00
Iteration 10/20: Best Distance = 10.00
Iteration 11/20: Best Distance = 10.00
Iteration 12/20: Best Distance = 10.00
Iteration 13/20: Best Distance = 10.00
Iteration 14/20: Best Distance = 10.00
Iteration 15/20: Best Distance = 10.00
Iteration 16/20: Best Distance = 10.00
Iteration 17/20: Best Distance = 10.00
Iteration 18/20: Best Distance = 10.00
Iteration 19/20: Best Distance = 10.00
Iteration 20/20: Best Distance = 10.00

OPTIMIZED OUTPUT
Shortest Route Found: A -> A
Minimum Total Travel Distance: 10 units
